## 1 · Imports, device, dataset

In [ ]:
from datasets import load_dataset

from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import math
import sentencepiece as spm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

ds = load_dataset("Helsinki-NLP/opus-100", "en-ja")


c:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda


## 2 · Hyperparameters & DataLoader

In [2]:
batch_size = 64
embedding_dim = 512
num_heads = 8
ff_hidden_dim = 2048
num_layers = 6
max_length = 256

train_loader = DataLoader(
    ds["train"],
    batch_size=batch_size,
    shuffle=True
)


## 3 · Write SentencePiece training corpus

In [3]:
with open("corpus.txt", "w", encoding="utf-8") as f:

    for item in ds["train"]:

        en = item["translation"]["en"]
        ja = item["translation"]["ja"]

        f.write(en + "\n")
        f.write(ja + "\n")


## 4 · Train SentencePiece model

In [4]:
spm.SentencePieceTrainer.train(
    input="corpus.txt",
    model_prefix="translator_sp",
    vocab_size=16000,
    model_type="Unigram",
    character_coverage=1.0,
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3
)


## 5 · Load SentencePiece model & verify special IDs

In [ ]:
sp = spm.SentencePieceProcessor()
sp.load("translator_sp.model")


assert sp.pad_id() == 0, (
    f"sp.pad_id() returned {sp.pad_id()}.  "
    "The SentencePiece model must be trained with pad_id=0. "
    "Re-run the SentencePieceTrainer cell and reload."
)
print(f"PAD={sp.pad_id()}  BOS={sp.bos_id()}  EOS={sp.eos_id()}  VOCAB={sp.get_piece_size()}")


PAD=0  BOS=2  EOS=3  VOCAB=16000


## 6 · Tokenize 

In [6]:
def tokenize(example):

    source = (
        [sp.bos_id()]
        + sp.encode(
            example["translation"]["en"],
            out_type=int
        )
        + [sp.eos_id()]
    )

    target = (
        [sp.bos_id()]
        + sp.encode(
            example["translation"]["ja"],
            out_type=int
        )
        + [sp.eos_id()]
    )

    return {
        "source": source,
        "target": target
    }


## 7 · Vocabulary 


In [7]:
VOCAB_SIZE = sp.get_piece_size()
PAD_ID     = sp.pad_id()
BOS_ID     = sp.bos_id()
EOS_ID     = sp.eos_id()

print(f"VOCAB_SIZE={VOCAB_SIZE}  PAD_ID={PAD_ID}  BOS_ID={BOS_ID}  EOS_ID={EOS_ID}")


VOCAB_SIZE=16000  PAD_ID=0  BOS_ID=2  EOS_ID=3


## 8 · MultiHeadAttention


In [ ]:
import torch
import torch.nn as nn


class MultiHeadAttention(nn.Module):

    def __init__(
        self,
        embedding_dim,
        num_heads
    ):
        super().__init__()

        assert embedding_dim % num_heads == 0

        self.embedding_dim = embedding_dim
        self.num_heads = num_heads
        self.head_dim = embedding_dim // num_heads

        self.query = nn.Linear(embedding_dim, embedding_dim)
        self.key   = nn.Linear(embedding_dim, embedding_dim)
        self.value = nn.Linear(embedding_dim, embedding_dim)
        self.fc_out = nn.Linear(embedding_dim, embedding_dim)

    def forward(
        self,
        query,
        key,
        value,
        mask=None   
    ):

        batch_size = query.shape[0]

        Q = self.query(query)
        K = self.key(key)
        V = self.value(value)

        Q = Q.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)

        if mask is not None:
           
            scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)

        attention = torch.softmax(scores, dim=-1)

        output = torch.matmul(attention, V)
        output = output.transpose(1, 2).contiguous()
        output = output.view(batch_size, -1, self.embedding_dim)
        output = self.fc_out(output)

        return output


## 9 · EncoderBlock

In [9]:
class EncoderBlock(nn.Module):

    def __init__(self, embedding_dim, num_heads, ff_hidden_dim):
        super().__init__()

        self.attention = MultiHeadAttention(embedding_dim, num_heads)

        self.norm1 = nn.LayerNorm(embedding_dim)
        self.norm2 = nn.LayerNorm(embedding_dim)

        self.ffn = nn.Sequential(
            nn.Linear(embedding_dim, ff_hidden_dim),
            nn.GELU(),
            nn.Linear(ff_hidden_dim, embedding_dim)
        )

    def forward(self, x, mask=None):

        attn = self.attention(x, x, x, mask)
        x = self.norm1(x + attn)

        ffn = self.ffn(x)
        x = self.norm2(x + ffn)

        return x


## 10 · DecoderBlock



In [ ]:
class DecoderBlock(nn.Module):

    def __init__(self, embedding_dim, num_heads, ff_hidden_dim):
        super().__init__()

        self.self_attention  = MultiHeadAttention(embedding_dim, num_heads)
        self.cross_attention = MultiHeadAttention(embedding_dim, num_heads)

        self.norm1 = nn.LayerNorm(embedding_dim)
        self.norm2 = nn.LayerNorm(embedding_dim)
        self.norm3 = nn.LayerNorm(embedding_dim)

        self.ffn = nn.Sequential(
            nn.Linear(embedding_dim, ff_hidden_dim),
            nn.GELU(),
            nn.Linear(ff_hidden_dim, embedding_dim)
        )

    def forward(self, x, encoder_output, tgt_mask=None, src_mask=None):

        
        attn = self.self_attention(x, x, x, tgt_mask)
        x = self.norm1(x + attn)

        
        cross = self.cross_attention(x, encoder_output, encoder_output, src_mask)
        x = self.norm2(x + cross)

        ffn = self.ffn(x)
        x = self.norm3(x + ffn)

        return x


## 11 · Encoder



In [ ]:
class Encoder(nn.Module):

    def __init__(self, embedding_dim, num_heads, ff_hidden_dim, num_layers):
        super().__init__()

        self.layers = nn.ModuleList([
            EncoderBlock(embedding_dim, num_heads, ff_hidden_dim)
            for _ in range(num_layers)
        ])

   
    def forward(self, x, mask=None):

        for layer in self.layers:
            x = layer(x, mask)

        return x


## 12 · Decoder



In [12]:
class Decoder(nn.Module):

    def __init__(self, embedding_dim, num_heads, ff_hidden_dim, num_layers):
        super().__init__()

        self.layers = nn.ModuleList([
            DecoderBlock(embedding_dim, num_heads, ff_hidden_dim)
            for _ in range(num_layers)
        ])

    def forward(self, x, encoder_output, tgt_mask, src_mask=None):

        for layer in self.layers:
            x = layer(x, encoder_output, tgt_mask, src_mask)

        return x


## 13 · causal_mask helper



In [ ]:
def causal_mask(seq_len, device):
    
    mask = torch.tril(
        torch.ones(seq_len, seq_len, device=device, dtype=torch.bool)
    )
    return mask.unsqueeze(0).unsqueeze(0)  


## 14 · Transformer



In [ ]:
class Transformer(nn.Module):

    def __init__(
        self,
        VOCAB_SIZE,
        embedding_dim,
        num_heads,
        ff_hidden_dim,
        num_layers,
        max_length
    ):
        super().__init__()

        self.max_length = max_length

        self.token_embedding    = nn.Embedding(VOCAB_SIZE, embedding_dim)
        self.position_embedding = nn.Embedding(max_length, embedding_dim)

        self.encoder = Encoder(embedding_dim, num_heads, ff_hidden_dim, num_layers)
        self.decoder = Decoder(embedding_dim, num_heads, ff_hidden_dim, num_layers)

        self.fc_out = nn.Linear(embedding_dim, VOCAB_SIZE)

    def forward(self, source_input, target_input, pad_id=0):

        B, src_len = source_input.shape
        _, tgt_len = target_input.shape

        
        src_pos = torch.arange(src_len, device=source_input.device).unsqueeze(0)
        src_pos = src_pos.clamp(max=self.max_length - 1)

        tgt_pos = torch.arange(tgt_len, device=target_input.device).unsqueeze(0)
        tgt_pos = tgt_pos.clamp(max=self.max_length - 1)

        src = self.token_embedding(source_input) + self.position_embedding(src_pos)
        tgt = self.token_embedding(target_input) + self.position_embedding(tgt_pos)

       
        src_key_padding_mask = (source_input != pad_id)            
        src_mask = src_key_padding_mask.unsqueeze(1).unsqueeze(2)  

        encoder_output = self.encoder(src, mask=src_mask)

        
        tgt_mask = causal_mask(tgt_len, device=target_input.device)  

        
        decoder_output = self.decoder(tgt, encoder_output, tgt_mask, src_mask)

        logits = self.fc_out(decoder_output)

        return logits


## 15 · Model, criterion, optimizer

In [15]:
model = Transformer(
    VOCAB_SIZE=VOCAB_SIZE,
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    ff_hidden_dim=ff_hidden_dim,
    num_layers=num_layers,
    max_length=max_length
).to(device)

pad_id = sp.pad_id()

criterion = nn.CrossEntropyLoss(ignore_index=pad_id)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-2
)


## 16 · AMP / GradScaler setup

In [16]:
import os
from torch.amp import autocast, GradScaler

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

scaler = GradScaler("cuda")


## 17 · Single-batch 



In [17]:
from torch.nn.utils.rnn import pad_sequence

batch = next(iter(train_loader))

source_texts = batch["translation"]["en"]

print(type(source_texts))
print(len(source_texts))

tokens = [
    torch.tensor(
        ([sp.bos_id()] + sp.encode(text, out_type=int) + [sp.eos_id()])[:max_length],
        dtype=torch.long
    )
    for text in source_texts
]

print("tokenization success")

padded = pad_sequence(
    tokens,
    batch_first=True,
    padding_value=sp.pad_id()
)

print("padding success")
print("shape       :", padded.shape)
print("max token   :", padded.max().item())
print("min token   :", padded.min().item())
print("seq_len     :", padded.shape[1], "(must be <=", max_length, ")")

# FIX: check sequence length, NOT just token values
assert padded.shape[1] <= max_length, (
    f"seq_len={padded.shape[1]} exceeds max_length={max_length} "
    "even after truncation – check the slicing logic above."
)
print("Sequence length check passed ✓")


<class 'list'>
64
tokenization success
padding success
shape       : torch.Size([64, 39])
max token   : 12162
min token   : 0
seq_len     : 39 (must be <= 256 )
Sequence length check passed ✓


## 18 · Token embedding smoke test

In [18]:
source_input = padded.to(device)

print("Moved to CUDA")

with torch.no_grad():
    emb = model.token_embedding(source_input)

print("Embedding success")
print(emb.shape)


Moved to CUDA
Embedding success
torch.Size([64, 39, 512])


## 19 · Position embedding smoke test

In [19]:
source_input = padded.to(device)
src_len = source_input.shape[1]

src_pos = torch.arange(src_len, device=source_input.device).unsqueeze(0)

print("Position indices range: 0 –", src_pos.max().item(), "(max_length-1 =", max_length - 1, ")")
assert src_pos.max().item() < max_length, (
    f"src_pos max={src_pos.max().item()} >= max_length={max_length}"
)

with torch.no_grad():
    token_emb = model.token_embedding(source_input)

print("Token embedding success")

with torch.no_grad():
    pos_emb = model.position_embedding(src_pos)

print("Position embedding success")

src = token_emb + pos_emb
print("Combined embedding success, shape:", src.shape)


Position indices range: 0 – 38 (max_length-1 = 255 )
Token embedding success
Position embedding success
Combined embedding success, shape: torch.Size([64, 39, 512])


## 20 · Encoder smoke test

In [20]:
with torch.no_grad():
    enc_out = model.encoder(src)

print("Encoder success")
print(enc_out.shape)
print("embedding_dim:", embedding_dim)
print("num_heads    :", num_heads)


Encoder success
torch.Size([64, 39, 512])
embedding_dim: 512
num_heads    : 8


## 21 · Training loop



In [ ]:
from tqdm import tqdm
from torch.amp import autocast, GradScaler
from torch.nn.utils.rnn import pad_sequence

print("SentencePiece vocab :", sp.get_piece_size())
print("Model vocab         :", model.token_embedding.num_embeddings)

assert sp.get_piece_size() == model.token_embedding.num_embeddings, \
    "SentencePiece vocab and model vocab do not match!"

epochs = 3
scaler = GradScaler("cuda")

for epoch in range(epochs):

    model.train()
    total_loss = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{epochs}"
    )

    for batch_idx, batch in enumerate(progress_bar):

        source_texts = batch["translation"]["en"]
        target_texts = batch["translation"]["ja"]

        
        source_tokens = [
            torch.tensor(
                ([sp.bos_id()] + sp.encode(text, out_type=int) + [sp.eos_id()])[:max_length],
                dtype=torch.long
            )
            for text in source_texts
        ]

        target_tokens = [
            torch.tensor(
                ([sp.bos_id()] + sp.encode(text, out_type=int) + [sp.eos_id()])[:max_length],
                dtype=torch.long
            )
            for text in target_texts
        ]

        source_input = pad_sequence(
            source_tokens,
            batch_first=True,
            padding_value=sp.pad_id()
        ).to(device)

        target_full = pad_sequence(
            target_tokens,
            batch_first=True,
            padding_value=sp.pad_id()
        ).to(device)

        target_input  = target_full[:, :-1]
        target_output = target_full[:, 1:]

        
        assert source_input.shape[1] <= max_length, (
            f"src_len={source_input.shape[1]} > max_length={max_length}"
        )
        assert target_input.shape[1] <= max_length, (
            f"tgt_len={target_input.shape[1]} > max_length={max_length}"
        )

        
        assert source_input.min() >= 0, \
            f"Negative source token: {source_input.min().item()}"
        assert target_input.min() >= 0, \
            f"Negative target token: {target_input.min().item()}"
        assert source_input.max() < VOCAB_SIZE, \
            f"SRC token {source_input.max().item()} >= VOCAB_SIZE {VOCAB_SIZE}"
        assert target_input.max() < VOCAB_SIZE, \
            f"TGT token {target_input.max().item()} >= VOCAB_SIZE {VOCAB_SIZE}"

        
        assert target_output.min() >= 0, \
            f"Negative value in target_output: {target_output.min().item()}"
        assert target_output.max() < VOCAB_SIZE, \
            f"target_output token {target_output.max().item()} >= VOCAB_SIZE {VOCAB_SIZE}"

        optimizer.zero_grad()

        with autocast("cuda"):

            predictions = model(
                source_input,
                target_input,
                pad_id=PAD_ID
            )

            loss = criterion(
                predictions.reshape(-1, VOCAB_SIZE),
                target_output.reshape(-1)
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        avg_loss = total_loss / (batch_idx + 1)
        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            avg_loss=f"{avg_loss:.4f}"
        )

    avg_epoch_loss = total_loss / len(train_loader)
    print(f"\nEpoch {epoch+1}/{epochs} Completed | Average Loss: {avg_epoch_loss:.4f}\n")

    torch.save(model.state_dict(), f"translator_epoch_{epoch+1}.pth")


SentencePiece vocab : 16000
Model vocab         : 16000


Epoch 1/3: 100%|██████████| 15625/15625 [25:58<00:00, 10.02it/s, avg_loss=4.4335, loss=3.5463] 



Epoch 1/3 Completed | Average Loss: 4.4335



Epoch 2/3: 100%|██████████| 15625/15625 [28:20<00:00,  9.19it/s, avg_loss=3.2044, loss=2.9539]  



Epoch 2/3 Completed | Average Loss: 3.2044



Epoch 3/3: 100%|██████████| 15625/15625 [27:26<00:00,  9.49it/s, avg_loss=2.7375, loss=2.6347] 



Epoch 3/3 Completed | Average Loss: 2.7375



## 22 · Post-training target_output check

In [22]:
print("VOCAB_SIZE:", VOCAB_SIZE)
print("Max target:", target_output.max().item())
print("Min target:", target_output.min().item())


VOCAB_SIZE: 16000
Max target: 14820
Min target: 0


## 23 · Memory cleanup

In [23]:
import gc
gc.collect()
torch.cuda.empty_cache()


## 24 · Inference: `translate()`



In [ ]:
def translate(
    text,
    model,
    sp,
    device,
    max_length=256
):

    model.eval()

    with torch.no_grad():

        
        source_tokens = (
            [sp.bos_id()]
            + sp.encode(text, out_type=int)
            + [sp.eos_id()]
        )[:max_length]

        source_input = torch.tensor(
            [source_tokens],
            dtype=torch.long,
            device=device
        )

        generated = torch.tensor(
            [[sp.bos_id()]],
            dtype=torch.long,
            device=device
        )

        for _ in range(max_length):

            output = model(source_input, generated, pad_id=sp.pad_id())

            next_token = output[:, -1, :].argmax(dim=-1)

            generated = torch.cat(
                [generated, next_token.unsqueeze(1)],
                dim=1
            )

            if next_token.item() == sp.eos_id():
                break

        ids = generated[0].tolist()
        ids = [
            x for x in ids
            if x not in [sp.pad_id(), sp.bos_id(), sp.eos_id()]
        ]

        return sp.decode(ids)


## 25 · Run a translation

In [29]:
import os

print(os.path.getsize("translator_epoch_3.pth")/(1024**2))

231.52801609039307


In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

english_text = "how are you ?"

translation = translate(
    text=english_text,
    model=model,
    sp=sp,
    device=device
)

print("\nTranslation:")
print(translation)



Translation:
気分は?
